# Virtual Mouse with Live Hand Motion Tracking

This notebook uses your webcam feed to move the system mouse pointer with your index finger and trigger clicks with pinch gestures.

## Install dependencies (run once)
```bash
pip install opencv-python mediapipe pyautogui numpy
```

> On Linux, you may also need: `sudo apt-get install python3-tk python3-dev` for `pyautogui` support.

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pyautogui

# Recommended so moving to the screen corner can still stop control quickly
pyautogui.FAILSAFE = True
pyautogui.PAUSE = 0

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

screen_w, screen_h = pyautogui.size()
print(f'Screen resolution: {screen_w}x{screen_h}')

In [ ]:
# Live Virtual Mouse
# Gestures:
# - Move mouse: index fingertip (landmark 8)
# - Left click: pinch index + thumb (8 and 4)
# - Right click: pinch middle + thumb (12 and 4)
# - Exit: press 'q' in the video window

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError('Could not open camera. Check camera permissions or device index.')

# Reduce capture size for smoother FPS
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# Smoothing state
prev_x, prev_y = 0, 0
smoothening = 5

# Avoid repeated click firing every frame
left_click_down = False
right_click_down = False

# Interaction area margin (ignore edge jitter)
frame_margin = 60

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    model_complexity=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
) as hands:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape

        # Draw active region
        cv2.rectangle(
            frame,
            (frame_margin, frame_margin),
            (w - frame_margin, h - frame_margin),
            (0, 255, 0),
            2
        )

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        if result.multi_hand_landmarks:
            hand_landmarks = result.multi_hand_landmarks[0]
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            lm = hand_landmarks.landmark

            # Convert normalized landmarks to pixels
            ix, iy = int(lm[8].x * w), int(lm[8].y * h)      # index tip
            tx, ty = int(lm[4].x * w), int(lm[4].y * h)      # thumb tip
            mx, my = int(lm[12].x * w), int(lm[12].y * h)    # middle tip

            # Show control points
            cv2.circle(frame, (ix, iy), 8, (255, 255, 0), -1)
            cv2.circle(frame, (tx, ty), 8, (0, 255, 255), -1)
            cv2.circle(frame, (mx, my), 8, (255, 0, 255), -1)

            # Map camera region -> full screen coords
            ix_clamped = np.clip(ix, frame_margin, w - frame_margin)
            iy_clamped = np.clip(iy, frame_margin, h - frame_margin)

            target_x = np.interp(ix_clamped, (frame_margin, w - frame_margin), (0, screen_w))
            target_y = np.interp(iy_clamped, (frame_margin, h - frame_margin), (0, screen_h))

            # Smooth cursor movement
            curr_x = prev_x + (target_x - prev_x) / smoothening
            curr_y = prev_y + (target_y - prev_y) / smoothening

            pyautogui.moveTo(curr_x, curr_y)
            prev_x, prev_y = curr_x, curr_y

            # Gesture distances
            dist_index_thumb = np.hypot(ix - tx, iy - ty)
            dist_middle_thumb = np.hypot(mx - tx, my - ty)

            # Left click pinch
            if dist_index_thumb < 30 and not left_click_down:
                pyautogui.click(button='left')
                left_click_down = True
                cv2.putText(frame, 'LEFT CLICK', (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            elif dist_index_thumb >= 35:
                left_click_down = False

            # Right click pinch
            if dist_middle_thumb < 30 and not right_click_down:
                pyautogui.click(button='right')
                right_click_down = True
                cv2.putText(frame, 'RIGHT CLICK', (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 200, 255), 2)
            elif dist_middle_thumb >= 35:
                right_click_down = False

        cv2.putText(frame, "Press 'q' to quit", (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.imshow('Virtual Mouse - Hand Tracking', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

## Tips to improve stability
- Keep your hand well-lit and fully visible in frame.
- Increase `smoothening` for steadier movement; decrease it for faster response.
- If clicks are too sensitive, lower the pinch threshold (e.g., `30 -> 25`).
- If your camera is not index `0`, try `cv2.VideoCapture(1)`.